# BART 모델을 활용한 뉴스 요약 Fine-tuning

본 실습 노트북은 교재의 실습 코드를 바탕으로 구현되었으며, **BART (Denoising Sequence-to-Sequence)** 논문의 아키텍처적 특성을 이해하고 실제 생성 요약 Tasks에 적용하는 것을 목표로 합니다.

1. **조건부 생성 (Conditional Generation):** BART는 인코더-디코더 전체 구조를 사용하여 입력 컨텍스트를 파악하고 새로운 문장을 생성합니다. 코드에서는 `BartForConditionalGeneration` 클래스를 사용합니다.
2. **손실 함수와 패딩 무시 (-100):** 생성 태스크에서 가변 길이 문장을 처리할 때 패딩 토큰은 손실(Loss) 계산에서 제외해야 합니다. 코드에서는 패딩 값으로 `-100`을 설정하여 Cross Entropy 손실 계산 시 자동 무시되도록 처리합니다.
3. **ROUGE 평가지표:** 생성된 요약문과 정답 요약문의 텍스트 유사도를 N-gram 정밀도 및 재현율 기반으로 평가합니다.


## 0. 필수 라이브러리 설치
데이터세트 로드 및 루지(ROUGE) 평가를 위한 Hugging Face 라이브러리들을 설치합니다.

In [1]:
!pip install datasets transformers evaluate rouge_score absl-py

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1552e448c43818fc310523787a07ab4a5f9207c9172e99d477d042fdfef9f816
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


## 1. 뉴스 요약 데이터세트 불러오기 및 분할 (교재 예제 7.18)
미국의 AI 기업 아르길라(Argilla)가 공개한 뉴스 요약 데이터세트를 불러와 학습, 검증, 테스트 데이터로 분리합니다. 연산 속도를 확보하기 위해 5,000개의 샘플만 추출하여 사용합니다.

In [2]:
import numpy as np
from datasets import load_dataset

# 데이터세트 불러오기
news = load_dataset("argilla/news-summary", split="test")
df = news.to_pandas().sample(5000, random_state=42)[["text", "prediction"]]

# 전처리: 중첩된 딕셔너리 구조에서 정답 텍스트만 추출
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])

# 6:2:2 비율로 데이터 분할 (학습: 3000, 검증: 1000, 테스트: 1000)
train, valid, test = np.split(
    df.sample(frac=1, random_state=42),
    [int(0.6 * len(df)), int(0.8 * len(df))]
)

print(f"Source News: {train.text.iloc[0][:200]}")
print(f"Summarization: {train.prediction.iloc[0][:50]}")
print(f"Training Data Size: {len(train)}")
print(f"Validation Data Size: {len(valid)}")
print(f"Testing Data Size: {len(test)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

data/train-00000-of-00001-ebc48879f34571(…):   0%|          | 0.00/1.54M [00:00<?, ?B/s]

data/test-00000-of-00001-6227bd8eb10a9b5(…):   0%|          | 0.00/31.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20417 [00:00<?, ? examples/s]

Source News: DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, well-educated
Summarization: Putin says had useful interaction with Trump at Vi
Training Data Size: 3000
Validation Data Size: 1000
Testing Data Size: 1000


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


## 2. BART 입력 텐서 및 데이터로더 생성 (교재 예제 7.19)
BART 토크나이저를 활용하여 입력 문장과 정답 요약문을 토큰화하고 패딩을 적용합니다.
**BART 논문 및 교재 핵심 개념:** 교차 엔트로피 손실 함수에서 패딩된 토큰을 무시하도록 레이블 패딩 값을 `-100`으로 설정합니다.

In [3]:
import torch
from transformers import BartTokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data, tokenizer, device):
    # 본문 텍스트 토큰화
    tokenized = tokenizer(
        data.text.tolist(),
        padding="longest",
        truncation=True,
        max_length=1024,
        return_tensors="pt"
    )

    labels = []
    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)

    # 요약문(정답) 토큰화
    for target in data.prediction:
        labels.append(tokenizer.encode(target, return_tensors="pt").squeeze())

    # 손실 함수 계산 시 패딩 무시를 위해 padding_value=-100 사용
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=-100).to(device)
    return TensorDataset(input_ids, attention_mask, padded_labels)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader

# 하이퍼파라미터 및 디바이스 설정
epochs = 3
batch_size = 8
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 토크나이저 초기화 및 데이터로더 생성
tokenizer = BartTokenizer.from_pretrained(pretrained_model_name_or_path="facebook/bart-base")

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print("데이터로더 구축 완료!")

Using device: cuda


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

데이터로더 구축 완료!


## 3. BART 조건부 생성 모델 및 최적화 함수 선언 (교재 예제 7.20)
조건부 생성 작업에 특화된 `BartForConditionalGeneration` 클래스를 사용해 6개 계층을 갖는 `facebook/bart-base` 모델을 인스턴스화합니다. 가중치 최적화를 위해 `AdamW` 알고리즘을 정의합니다.

In [4]:
from torch import optim
from transformers import BartForConditionalGeneration

# 모델 불러오기 및 GPU 배치
model = BartForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
).to(device)

# 최적화 알고리즘 설정 (Learning Rate = 5e-5)
optimizer = optim.AdamW(model.parameters(), lr=5e-5, eps=1e-8)

print("BART 모델 및 옵티마이저 선언 완료!")

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

BART 모델 및 옵티마이저 선언 완료!


## 4. 모델 학습 및 평가 루프 구성 (교재 예제 7.21 기반)
교재에서 수식화된 ROUGE-2 평가지표 계산 함수(`calc_rouge`)와 검증 루프(`evaluation`)를 포함하여, 전체 3 에포크 동안 파인 튜닝을 진행하는 전체 학습 파이프라인 코드입니다.


In [5]:
import evaluate

# 허깅페이스 evaluate 라이브러리에서 ROUGE 메트릭 로드
rouge_score = evaluate.load("rouge", tokenizer=tokenizer)

def calc_rouge(preds, labels):
    # 확률 스코어가 가장 높은 토큰 인덱스 추출
    preds = preds.argmax(axis=-1)

    # -100 패딩 값을 디코딩이 가능하도록 토크나이저의 pad_token_id로 복원
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # 정수 토큰 배열을 실제 텍스트 문자열로 변환
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE 점수 계산
    rouge2 = rouge_score.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    return rouge2["rouge2"]

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval() # 평가 모드 전환
        val_loss, val_rouge = 0.0, 0.0

        for input_ids, attention_mask, labels in dataloader:
            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            logits = outputs.logits
            loss = outputs.loss

            # 데이터 이동 및 넘파이 변환
            logits = logits.detach().cpu().numpy()
            label_ids = labels.to("cpu").numpy()

            rouge = calc_rouge(logits, label_ids)
            val_loss += loss.item()
            val_rouge += rouge

        val_loss = val_loss / len(dataloader)
        val_rouge = val_rouge / len(dataloader)
        return val_loss, val_rouge

# --- 본격적인 모델 학습 (Training Loop) Run ---
import os
os.makedirs("../models", exist_ok=True)
best_rouge = 0.0

print("🚀 BART 미세 조정 학습을 시작합니다. (약 20~30분 소요)")
for epoch in range(epochs):
    model.train() # 학습 모드 전환
    total_train_loss = 0.0

    for step, (input_ids, attention_mask, labels) in enumerate(train_dataloader):
        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        if (step + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Step [{step+1}/{len(train_dataloader)}] | Loss: {loss.item():.4f}")

    # 에포크 종료 후 평가 수행
    avg_train_loss = total_train_loss / len(train_dataloader)
    val_loss, val_rouge = evaluation(model, valid_dataloader)

    print(f"\n[Epoch {epoch+1} 결과] Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Rouge-2: {val_rouge:.4f}")

    # 최고 성능 모델 저장
    if val_rouge > best_rouge:
        best_rouge = val_rouge
        torch.save(model.state_dict(), "../models/BartForConditionalGeneration.pt")
        print("⭐ 최고 성능 갱신! 모델 가중치 저장 완료.\n")

🚀 BART 미세 조정 학습을 시작합니다. (약 20~30분 소요)
Epoch [1/3] | Step [100/375] | Loss: 2.4253
Epoch [1/3] | Step [200/375] | Loss: 1.7921
Epoch [1/3] | Step [300/375] | Loss: 2.4110

[Epoch 1 결과] Train Loss: 2.1584 | Val Loss: 1.8755 | Val Rouge-2: 0.2599
⭐ 최고 성능 갱신! 모델 가중치 저장 완료.

Epoch [2/3] | Step [100/375] | Loss: 1.6786
Epoch [2/3] | Step [200/375] | Loss: 1.8839
Epoch [2/3] | Step [300/375] | Loss: 1.3839

[Epoch 2 결과] Train Loss: 1.6043 | Val Loss: 1.8807 | Val Rouge-2: 0.2626
⭐ 최고 성능 갱신! 모델 가중치 저장 완료.

Epoch [3/3] | Step [100/375] | Loss: 1.2133
Epoch [3/3] | Step [200/375] | Loss: 1.0791
Epoch [3/3] | Step [300/375] | Loss: 1.3840

[Epoch 3 결과] Train Loss: 1.2329 | Val Loss: 2.0361 | Val Rouge-2: 0.2505


## 5. 최종 테스트 데이터세트 평가 (교재 예제 7.22)
학습 과정에서 검증 세트 기준 성능이 가장 우수했던 모델 가중치를 로드하여, 학습에 쓰이지 않은 최종 테스트 데이터를 대상으로 평가를 수행합니다.

In [6]:
# 저장된 최고의 가중치 가동
model = BartForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
).to(device)
model.load_state_dict(torch.load("../models/BartForConditionalGeneration.pt"))

# 최종 평가 실행
test_loss, test_rouge_score = evaluation(model, test_dataloader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test ROUGE-2 Score: {test_rouge_score:.4f}")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Test Loss: 1.8353
Test ROUGE-2 Score: 0.2667


## 6. 모델 요약 생성 문장 실제 비교 (교재 예제 7.23)
Hugging Face의 `pipeline` 추론 함수를 활용해 실제 뉴스 원문(Test 데이터)에 대한 요약문을 생성하고, 이를 정답 요약문과 직접 눈으로 비교하며 품질을 확인합니다.

In [8]:
import transformers
print(transformers.__version__)

5.10.2


In [7]:
from transformers import pipeline

# 요약 파이프라인 정의 (인퍼런스를 위해 디바이스를 cpu 혹은 gpu로 매핑 가능)
summarizer = pipeline(
    task="summarization",
    model=model,
    tokenizer=tokenizer,
    max_length=54,
    device=0 if torch.cuda.is_available() else -1
)

# 상위 5개 데이터 추출 및 비교 분석
for index in range(5):
    news_text = test.text.iloc[index]
    summarization = test.prediction.iloc[index]
    predicted_summarization = summarizer(news_text)[0]["summary_text"]

    print(f"[{index+1}번 뉴스 요약 비교]")
    print(f"정답 요약문 : {summarization}")
    print(f"모델 요약문 : {predicted_summarization}")
    print("-" * 50)

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"